### 1. Install deps

In [1]:
import sys
# Клонуємо актуальний репозиторій
!git clone https://github.com/dmytroslav/NLP_Course.git
%cd NLP_Course
!pip install spacy pandas scikit-learn
!python -m spacy download uk_core_news_sm

Cloning into 'NLP_Course'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 59 (delta 16), reused 53 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 1.90 MiB | 10.83 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/NLP_Course
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 102.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('uk_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime'

### 2. Data access

In [2]:
import pandas as pd
import spacy
import sys
import os

sys.path.append(os.path.abspath('src'))
from ling_features import extract_ling_features

# Завантажуємо дані з попередньої лаби
df = pd.read_csv('data/processed_v2.csv')

# Видаляємо порожні рядки, якщо такі раптом виникли
df = df.dropna(subset=['text'])
print(f"Завантажено {len(df)} текстів для аналізу.")

Завантажено 10735 текстів для аналізу.


### 3. Lemma/POS extraction

In [3]:
print("Завантаження моделі spaCy...")
nlp = spacy.load("uk_core_news_sm")

print("Генерація лем та POS-тегів (це займе 2-3 хвилини)...")
# Застосовуємо функцію до кожного рядка
extracted = df['text'].apply(lambda x: extract_ling_features(str(x), nlp))

# Розширюємо датафрейм новими колонками
df['lemma_text'] = extracted.apply(lambda x: x['lemma_text'])
df['pos_seq'] = extracted.apply(lambda x: x['pos_seq'])

print("Генерацію завершено!")
display(df[['text', 'lemma_text', 'pos_seq']].head(3))

Завантаження моделі spaCy...
Генерація лем та POS-тегів (це займе 2-3 хвилини)...
Генерацію завершено!


,text,lemma_text,pos_seq
0,"Просто слухайте цей діалог. Ні, це не нарізка ...","просто слухати цей діалог . ні , це не нарізка...",PART VERB DET NOUN PUNCT INTJ PUNCT PRON PART ...
1,️ Рубль став найнестабільнішою валютою у всьом...,️ рубль стати найнестабільніший валюта у ввесь...,SYM NOUN VERB ADJ NOUN ADP DET NOUN
2,Перше звернення мера Мелітополя Івана Федорова...,перший звернення мер мелітополь іван федоров п...,ADJ NOUN NOUN PROPN PROPN PROPN ADP NOUN


### 4. Baseline comparison (TF-IDF + Logistic Regression)

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Підготовка даних
X_raw = df['text']
X_lemma = df['lemma_text']
y = df['label'] # або 'Label', перевір назву колонки у своєму файлі!

# Розбиття на train/test
X_raw_train, X_raw_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42)
X_lem_train, X_lem_test, _, _ = train_test_split(X_lemma, y, test_size=0.2, random_state=42)

def train_and_evaluate(X_train, X_test, y_train, y_test, name):
    vectorizer = TfidfVectorizer(max_features=5000)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf.fit(X_train_vec, y_train)

    preds = clf.predict(X_test_vec)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='macro')

    print(f"--- Результати для {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro-F1: {f1:.4f}\n")
    return acc, f1

print("Навчання Baseline 1 (Original Text)...")
acc_raw, f1_raw = train_and_evaluate(X_raw_train, X_raw_test, y_train, y_test, "Processed Text (No Lemmas)")

print("Навчання Baseline 2 (Lemmatized Text)...")
acc_lem, f1_lem = train_and_evaluate(X_lem_train, X_lem_test, y_train, y_test, "Lemma Text")

Навчання Baseline 1 (Original Text)...
--- Результати для Processed Text (No Lemmas) ---
Accuracy: 0.8882
Macro-F1: 0.8442

Навчання Baseline 2 (Lemmatized Text)...
--- Результати для Lemma Text ---
Accuracy: 0.8803
Macro-F1: 0.8347



### 5. Error analysis (Помилки лематизації)

In [5]:
# Демонстрація помилок лематизатора (10 прикладів)
errors = [
    "Русні п*зда!", # Нецензурщина/Сленг
    "Світло відключать по вул. Шевченка", # Скорочення
    "Шо там у вас?", # Суржик
    "HIMARS приїхали в Україну", # Англіцизми
    "Бавовна на росії", # Іронічний сленг
    "кремль горить", # З маленької літери власна назва
    "ДТП сталося зранку", # Абревіатури
    "Він айтішник", # Неологізм
    "Зеленський дав інтерв'ю", # Апостроф
    "ппц як холодно" # Інтернет-сленг
]

print("--- Error Analysis ---")
for text in errors:
    doc = nlp(text)
    lemmas = [t.lemma_ for t in doc]
    print(f"Original: {text}")
    print(f"Lemmas:   {' '.join(lemmas)}")
    print("-" * 30)

--- Error Analysis ---
Original: Русні п*зда!
Lemmas:   русні п*зда !
------------------------------
Original: Світло відключать по вул. Шевченка
Lemmas:   світло відключати по вул. шевченко
------------------------------
Original: Шо там у вас?
Lemmas:   шо там у ви ?
------------------------------
Original: HIMARS приїхали в Україну
Lemmas:   himars приїхати в україна
------------------------------
Original: Бавовна на росії
Lemmas:   бавовна на росія
------------------------------
Original: кремль горить
Lemmas:   кремль горіти
------------------------------
Original: ДТП сталося зранку
Lemmas:   дтп статися зранку
------------------------------
Original: Він айтішник
Lemmas:   він айтішник
------------------------------
Original: Зеленський дав інтерв'ю
Lemmas:   зеленський дати інтерв'ю
------------------------------
Original: ппц як холодно
Lemmas:   ппц як холодно
------------------------------


### 6. Генерація docs/audit_summary_lab3.md

In [6]:
diff_f1 = f1_lem - f1_raw
conclusion = ""

if diff_f1 > 0.01:
    conclusion = "Лематизація суттєво покращила модель. Використовуємо леми."
elif diff_f1 < -0.01:
    conclusion = "Лематизація погіршила результат (ймовірно, через втрату стилістичних маркерів фейків). Леми не беремо для класифікатора."
else:
    conclusion = "Лематизація не дала значного приросту. Для економії ресурсів можна використовувати звичайний текст, або залишити леми для retrieval."

audit_content = f"""# Data Audit Summary (Lab 3)

## Порівняння моделей (TF-IDF + Logistic Regression)
- **Baseline 1 (Processed Text):**
  - Accuracy: {acc_raw:.4f}
  - Macro-F1: {f1_raw:.4f}

- **Baseline 2 (Lemma Text):**
  - Accuracy: {acc_lem:.4f}
  - Macro-F1: {f1_lem:.4f}

## Error Analysis (Типові помилки spaCy на наших даних)
1. **Суржик та сленг:** Слова типу "шо", "русня", "бавовна" часто не розпізнаються коректно і залишаються у вихідній формі, або лематизатор намагається "вгадати" корінь.
2. **Абревіатури:** "ЗСУ", "ДТП" можуть втрачати регістр або отримувати дивні леми, якщо контекст нечіткий.
3. **Англіцизми:** Нові слова ("айтішник", "HIMARS") сприймаються як невідомі іменники (NOUN) без зміни форми.
*Чи це критично?* Для нашої задачі класифікації фейків специфічний сленг та суржик є важливими маркерами емоційності. Якщо лематизатор їх "ламає", модель може втратити сигнал.

## Висновок
{conclusion}
"""

with open('docs/audit_summary_lab3.md', 'w', encoding='utf-8') as f:
    f.write(audit_content)

print("Файл docs/audit_summary_lab3.md згенеровано. Висновок:")
print(conclusion)

Файл docs/audit_summary_lab3.md згенеровано. Висновок:
Лематизація не дала значного приросту. Для економії ресурсів можна використовувати звичайний текст, або залишити леми для retrieval.
